# Phase 2 — MongoDB NoSQL Storage

This notebook demonstrates the NoSQL storage layer.

MongoDB stores:

1. `user_profiles`
2. `item_pairs`
3. `product_catalog`

MongoDB was chosen because the project needs fast lookup of user profiles and product category data for recommendation decisions.

In [8]:
from pymongo import MongoClient
import pandas as pd

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "ecommerce_recs"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

In [9]:
collection_counts = {
    "user_profiles": db.user_profiles.count_documents({}),
    "item_pairs": db.item_pairs.count_documents({}),
    "product_catalog": db.product_catalog.count_documents({})
}

pd.DataFrame(collection_counts.items(), columns=["Collection", "Document Count"])

,Collection,Document Count
0,user_profiles,25000
1,item_pairs,50000
2,product_catalog,5000


## User Profiles Collection

Each document stores one user's top product categories based on Spark affinity scoring.

In [10]:
user_profile = db.user_profiles.find_one({"user_id": "User_0"}, {"_id": 0})
user_profile

{'user_id': 'User_0',
 'top_categories': [{'category': 'Home', 'score': 150},
  {'category': 'Clothing', 'score': 124},
  {'category': 'Books', 'score': 113}]}

In [11]:
pd.DataFrame(user_profile["top_categories"])

,category,score
0,Home,150
1,Clothing,124
2,Books,113


## Product Catalog Collection

The product catalog is used to look up the category and brand of a selected item.

In [12]:
product_1 = db.product_catalog.find_one({"item_id": "ITEM_1911"}, {"_id": 0})
product_2 = db.product_catalog.find_one({"item_id": "ITEM_1618"}, {"_id": 0})

pd.DataFrame([product_1, product_2])

,item_id,category,brand
0,ITEM_1911,Clothing,Zara
1,ITEM_1618,Electronics,Dell


## Item Pairs Collection

The `item_pairs` collection stores market basket results generated by Spark.

In [13]:
top_item_pairs = list(
    db.item_pairs.find({}, {"_id": 0})
    .sort("pair_count", -1)
    .limit(20)
)

pd.DataFrame(top_item_pairs)

,item_1,item_2,pair_count
0,ITEM_2165,ITEM_3750,2
1,ITEM_3122,ITEM_4731,2
2,ITEM_301,ITEM_3891,2
3,ITEM_291,ITEM_74,2
4,ITEM_148,ITEM_2945,2
5,ITEM_1060,ITEM_2214,2
6,ITEM_2436,ITEM_297,2
7,ITEM_386,ITEM_4168,2
8,ITEM_1116,ITEM_4826,2
9,ITEM_1446,ITEM_2643,2


## Indexes

Indexes were created to support fast lookup.

print("user_profiles indexes:")
print(db.user_profiles.index_information())

print("\nproduct_catalog indexes:")
print(db.product_catalog.index_information())

print("\nitem_pairs indexes:")
print(db.item_pairs.index_information())

In [14]:
client.close()